In [ ]:
# Importing libraries
import pandas as pd
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler 
import warnings
warnings.filterwarnings('ignore')
import scipy
from sklearn.cluster import KMeans
from PIL import Image
import statsmodels.api
from itertools import combinations

In [ ]:
# Importing coordinates
local = pd.read_csv(r"C:\Users\joaoa\Desktop\Doutoramento\Professora\Nova pasta\Todos os Pontos (temperatura)\local.cvs")

# Defining parameters
min_corr = 0.1 # 0.6
Y = range(1950, 2023) # 1950-2023
K = range(3, 4) # 2-7
T = ["no restriction"] # ["no restriction", "9h-9h"]
H = [1,2,4,6,12,24,48] # [1,2,3,4,6,8,12,24,48,72,96,120]

In [ ]:
max_precipitations = pd.read_csv(r"D:\Precipitation\by_water_year\max_precipitations_(mm).csv")
max_precipitations

In [ ]:
# Collecting all information in one dictionary
dict_max = {"no restriction" : {}, "9h-9h" : {}}

for type_of_period in T: 
    for hours in H:
        dict_max[type_of_period][str(hours) + "h"] = max_precipitations[(max_precipitations["type_of_period"] == type_of_period) & (max_precipitations["duration"] == str(hours) + "h")].iloc[:, 3:].reset_index(drop = True)

In [ ]:
seq_clusters = {"no restriction" : {}, "9h-9h" : {}}

for t in T:

    for n in K:
        seq_clusters[t][n] = pd.DataFrame()

    for h in H:
        df = dict_max[t][str(h) + "h"]

        # Standardizing dataframe
        scaler = StandardScaler() 
        variables = scaler.fit_transform(df) 
        variables = pd.DataFrame(variables, columns = df.columns)
        variables = variables*np.sqrt((len(variables)-1)/len(variables))
        var_list = variables.columns

        # Transposing dataframe
        transp = variables.transpose()

        # Clustering (PCA)
        for n in K:
            pca = PCA(n_components = n)
            scores = pca.fit_transform(variables.values)
            print("Time type: " + t + " | Time interval: " + str(h) + "h | Clusters: " + str(n) + " | Explained variance: " + str(round(sum(pca.explained_variance_ratio_)*100,1)) + " %")


            comp_list = []
            comp_list_num = []
            for i in range(1, len(pca.explained_variance_)+1):
                comp_list.append("PC_" + str(i))
                comp_list_num.append(str(i))

            scores = pd.DataFrame(scores, columns = comp_list)
            eigenvalues = pd.DataFrame(pca.explained_variance_, index = comp_list, columns = ["explained variance"])
            eigenvectors = pd.DataFrame(np.transpose(pca.components_), index = var_list, columns = comp_list)
            loadings = pd.DataFrame(index = var_list, columns = comp_list)
            for col in range(0,len(loadings.columns)):
                loadings.iloc[:,col] = eigenvectors.iloc[:,col]*np.sqrt(eigenvalues.values[col])
            rotated_loadings = pd.DataFrame(statsmodels.multivariate.factor_rotation.rotate_factors(loadings.values, "varimax")[0], index = var_list, columns = comp_list_num)
            rotated_loadings = abs(rotated_loadings)
            rotated_loadings["max"] = rotated_loadings.max(axis = 1)
            
            # Checking if max correlation of each point is higher than or equal to 1012
            if len(rotated_loadings[rotated_loadings["max"] >= min_corr]) != 1012:
                raise ValueError("Current correlation is not enough to work with all the points")
                
            rotated_loadings["cluster"] = ""
            for row in range(0,len(rotated_loadings)):
                for col in range(0,n):
                    if rotated_loadings.iloc[row,col] >= min_corr:
                        rotated_loadings.iloc[row,-1] += str(col+1)
            transp["clusters_" + str(n) + " | " + str(h) + "h (" + t + ")"] = rotated_loadings["cluster"]           
            
            # Concatenating clusters   
            seq_clusters[t][n] = pd.concat([seq_clusters[t][n], transp.iloc[:,-1]], axis = 1)
            
    # Creating column "point"
    for n in K:
        seq_clusters[t][n].insert(0, "point", seq_clusters[t][n].index.values)
        seq_clusters[t][n]["point"] = seq_clusters[t][n]["point"].str.replace("P", "")
        seq_clusters[t][n] = seq_clusters[t][n].reset_index(drop = True)
    check = seq_clusters[t][n].iloc[:,0].values
        
    # Creating all combinations with corr >= corr_min
    seq_list = {}
    for row in range(0,1012):
        print("Ponto", row)
        p = seq_clusters[t][n].iloc[row,0]
        lista_antiga = [[p]]

        for col in range(1,len(seq_clusters[t][n].columns)):
            lista_nova = []

            for num in seq_clusters[t][n].iloc[row,col]:

                for item in lista_antiga:
                    lista_nova.append(item + [num])

            lista_antiga = lista_nova

        seq_list[p] = lista_antiga
        for seq in seq_list[p]:
            seq_clusters[t][n].loc[len(seq_clusters[t][n])] = seq

    seq_clusters[t][n] = seq_clusters[t][n].iloc[1012:,:]
    seq_clusters[t][n] = seq_clusters[t][n].reset_index(drop = True)

    # Creating a final column with sequence
    for n in K:

        i = 1
        seq_clusters[t][n]["code"] = ""
        for h in H:
            seq_clusters[t][n]["code"] = seq_clusters[t][n]["code"] + seq_clusters[t][n].iloc[:,i]
            i += 1

In [ ]:
seq_clusters["no restriction"][3]

In [ ]:
a = seq_clusters["no restriction"][3].iloc[:,0].values
a

In [ ]:
# Deciding most frequent exclusive clusters 
commun_points = {"no restriction" : {}, "9h-9h" : {}}

for n in K:
    for t in T:
        
        # Saving all codes with absolute frequency greater than or equal to 5
        i = 0
        codes = seq_clusters[t][n].groupby("code")["code"].agg("count").sort_values(ascending = False)
        for _ in codes:
            i += 1
            if codes[i] < 250: ############################ 1012/(2*n)
                codes = codes[0:i]
                break

        print("Number of codes", len(codes)) ################# não precisa de estar aqui
        # Creating all possible combinations of clusters
        count_list = list(range(0,len(codes)))
        comb = list(combinations(count_list, n))

        for _ in range(0, len(comb)):
            comb[_] = list(comb[_])
            comb[_].insert(0, 0)
            for element in comb[_][1:]:
                comb[_][0] += codes[element]

        elemination = []
        for hip in comb:
            if hip[0] != 1012:
                elemination.append(hip)

        for hip in elemination:
            comb.remove(hip)
        print("Number of combinations", len(comb)) ################# não precisa de estar aqui



        # Finding the combination with the highest number of points
        for hip in comb:
            top_lista = []
            aux = seq_clusters[t][n].copy()

            for _ in hip[1:]:
                top_lista.append(codes.index[_])

            aux = aux[aux["code"].isin(top_lista)]
            for top_code in top_lista:
                for index in range(0,len(top_code)):
                    aux = aux[(aux.iloc[:, index] != top_code[index]) | (aux.loc[:, "code"] == top_code)]

            if aux.iloc[:,0].values == check:
                print("Conseguimos") ################################## não é preciso
                break
        
        # Creating a commun column
        seq_clusters[t][n]["commun"] = "-1"

        for _ in range(0, len(top_lista)):
            seq_clusters[t][n].loc[seq_clusters[t][n]["code"] == top_lista[_], "commun"] = str(_)
            
        
        seq_clusters[t][n] = seq_clusters[t][n][seq_clusters[t][n]["commun"] != "-1"]

In [ ]:
codes

In [ ]:
comb

In [ ]:
seq_clusters["no restriction"][3]

In [ ]:
# Folder that stores the saved plots
folder = r"C:\Users\joaoa\Desktop\plots_3"
    
# Visualization
for n in K:
    for t in T:

        seq_clusters[t][n] = seq_clusters[t][n].reset_index(drop = True)
        df_plot = pd.concat([local, seq_clusters[t][n]], axis = 1)

        # Plotting clusters
        for h in H:
            
            if (n <= 3 and h <= 24) or (n == 2 and t == "9h-9h" and h == 48) or (n == 5 and t == "9h-9h" and h == 8) or (n == 6 and t == "9h-9h" and h in [3,4]) or (n == 4 and (h <= 12 or  (t == "no restriction" and h == 24))):
                palette_1 = ["lightgrey", "royalblue", "forestgreen", "saddlebrown", "orange", "red", "hotpink"]
            elif n == 2 and h == 120:
                palette_1 = ["royalblue", "forestgreen", "lightgrey", "saddlebrown", "orange", "red", "hotpink"]
            elif n == 4 and h <= 4 and t == "no restriction":
                palette_1 = ["lightgrey", "royalblue", "orange", "forestgreen", "saddlebrown", "red", "hotpink"]
            elif n == 4 and h == 6 and t == "no restriction":
                palette_1 = ["lightgrey", "royalblue", "orange", "saddlebrown", "forestgreen", "red", "hotpink"]
            elif (n == 4 and h <= 4) or (n == 6 and t == "9h-9h" and h == 6):
                palette_1 = ["lightgrey", "royalblue", "forestgreen", "orange", "saddlebrown", "red", "hotpink"]
            elif n == 4 and t == "9h-9h" and h == 24:
                palette_1 = ["forestgreen", "lightgrey", "royalblue", "saddlebrown", "orange", "red", "hotpink"]
            elif n == 5 and h <= 2:
                palette_1 = ["royalblue", "lightgrey", "forestgreen", "saddlebrown", "red", "orange", "hotpink"]
            elif n == 5 and h <= 6:
                palette_1 = ["lightgrey", "royalblue", "forestgreen", "saddlebrown", "red", "orange", "hotpink"]
                

            elif n == 6 and t == "no restriction" and h in [3,4,6]:
                palette_1 = ["lightgrey", "royalblue", "forestgreen", "saddlebrown", "orange", "red", "hotpink"] 
            elif n == 6 and t == "no restriction" and h == 120:
                palette_1 = ["royalblue", "forestgreen", "lightgrey", "saddlebrown", "red", "hotpink", "orange"]
            elif n == 6 and t == "no restriction" and h == 96:
                palette_1 = ["royalblue", "forestgreen", "lightgrey", "saddlebrown", "orange", "red", "hotpink"]
            elif n == 6 and t == "no restriction" and h in [24,48]:
                palette_1 = ["royalblue", "lightgrey", "forestgreen", "saddlebrown", "red", "hotpink", "orange"]
            elif n == 6 and t == "no restriction" and h <= 2:
                palette_1 = ["royalblue", "lightgrey", "forestgreen", "saddlebrown", "orange", "red", "hotpink"]
            elif n == 6 and t == "no restriction" and h == 12:
                palette_1 = ["royalblue", "lightgrey", "forestgreen", "saddlebrown", "red", "hotpink", "orange"]
            elif n == 6 and t == "no restriction" and h in [8,72]:
                palette_1 = ["royalblue", "lightgrey", "forestgreen", "saddlebrown", "red", "orange", "hotpink"]
                
                
            elif n == 6 and t == "9h-9h" and h in [72,120]:
                palette_1 = ["royalblue", "forestgreen", "lightgrey", "orange", "red", "hotpink", "saddlebrown"]
            elif n == 6 and t == "9h-9h" and h in [48,96]:
                palette_1 = ["royalblue", "lightgrey", "forestgreen", "orange", "red", "hotpink", "saddlebrown"]
            elif n == 6 and t == "9h-9h" and h in [8,24]:
                palette_1 = ["royalblue", "lightgrey", "forestgreen", "orange", "saddlebrown", "red", "hotpink"]
            elif n == 6 and t == "9h-9h" and h == 12:
                palette_1 = ["royalblue", "lightgrey", "forestgreen", "orange", "red", "saddlebrown", "hotpink"]
                
                
                
            else:
                palette_1 = ["royalblue", "lightgrey", "forestgreen", "saddlebrown", "orange", "red", "hotpink"]
            
            
            sns.set_theme(style = 'white')
            sns.relplot(data = df_plot, x = "longitude", y = "latitude", hue = "clusters_" + str(n) + " | " + str(h) + "h (" + t + ")", palette = palette_1, marker = "s", height = 6, aspect = 0.7, legend = False).set(title = "PCA | " + str(n) + " clusters" + " | " + str(h) + "h (" + t + ")")
            plt.axis("off")
            plt.savefig(folder + "\pca_" + str(n) + "_clusters_" + str(h) + "h_(" + t + ")_b.png", bbox_inches = 'tight')
            plt.show()

        # Plotting commun area
        
        if n == 4 and t == "9h-9h":
            palette_2 = ["lightgrey", "royalblue", "saddlebrown", "orange", "forestgreen", "red", "hotpink"]
        elif n == 4 and t == "no restriction":
            palette_2 = ["lightgrey", "royalblue", "saddlebrown", "forestgreen", "orange", "red", "hotpink"]
        elif n == 5:
            palette_2 = ["lightgrey", "royalblue", "saddlebrown", "orange", "red", "forestgreen", "hotpink"]
        elif n == 6 and t == "no restriction":
            palette_2 = ["lightgrey", "royalblue", "saddlebrown", "red", "hotpink", "forestgreen", "orange"]
        elif n == 6 and t == "9h-9h":
            palette_2 = ["lightgrey", "royalblue", "orange", "red", "hotpink", "forestgreen",  "saddlebrown"]
        elif n == 3 and t == "no restriction":
            palette_2 = ["royalblue", "lightgrey", "forestgreen",  "saddlebrown", "orange", "red", "hotpink"]
        else:
            palette_2 = ["lightgrey", "royalblue", "forestgreen",  "saddlebrown", "orange", "red", "hotpink"]
            
            
        sns.set_theme(style = 'white')
        sns.relplot(data = df_plot, x = "longitude", y = "latitude", hue = "commun", palette = palette_2, marker = "s", height = 6, aspect = 0.7, legend = False).set(title = "PCA | " + str(n) + " clusters" + " | " + str(commun_points[t][n]) + " pts (" + t + ")")
        plt.axis("off")
        plt.savefig(folder + "\pca_" + str(n) + "_clusters_commun_(" + t + ")_b.png", bbox_inches = 'tight')
        plt.show()

In [ ]:
for n in K:
    for t in T:

        seq_clusters[t][n] = seq_clusters[t][n].reset_index(drop = True)
        df_plot = pd.concat([local, seq_clusters[t][n]], axis = 1)

        fig = plt.figure(figsize = (55, 55))
        columns = len(H) + 1
        rows = 1
        i = 0

        for h in H:
            i += 1
            img = Image.open(folder + "\pca_" + str(n) + "_clusters_" + str(h) + "h_(" + t + ")_b.png")
            fig.add_subplot(rows, columns, i)
            plt.subplots_adjust(wspace = 0, hspace = 0)
            plt.axis("off")
            plt.imshow(img)

        i += 1
        img = Image.open(folder + "\pca_" + str(n) + "_clusters_commun_(" + t + ")_b.png")
        fig.add_subplot(rows, columns, i)
        plt.subplots_adjust(wspace = 0, hspace = 0)
        plt.axis("off")
        plt.imshow(img)
        
        print(str(n) + " clusters   |   " + t + "   |   Number of commun points:", commun_points[t][n])
        plt.savefig(folder + "\pca_" + str(n) + "_clusters_all_maps_(" + t + ")_b.png", bbox_inches = 'tight')
        plt.show()